In [2]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM


if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"GPU Available: {gpu_name} | Total VRAM: {gpu_mem:.2f} GB")
    device = "cuda"
else:
    print("No CUDA GPU detected. Running on CPU (using lightweight FP32/BF16 LoRA).")
    device = "cpu"


model_id = "Qwen/Qwen2.5-0.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_id)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32,
    device_map="auto" if device == "cuda" else None
)
if device == "cpu":
    model = model.to("cpu")

print(f"Loaded {model_id} successfully!")


def ask_model(target_model, question):
    messages = [
        {"role": "system", "content": "You are a helpful customer support assistant for TechZone Computer Shop."},
        {"role": "user", "content": question}
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    with torch.no_grad():
        outputs = target_model.generate(
            **inputs,
            max_new_tokens=100,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )
    response = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    return response.strip()


test_questions = [
    "What are TechZone Computer Shop's opening hours?",                         # Seen in training
    "What is TechZone's warranty policy on custom-built gaming PCs?",          # Seen in training
    "Do you offer student discounts on laptops at TechZone?",                  # Unseen question
    "Can I bring my broken monitor to TechZone for same-day screen repair?"    # Unseen question
]

print("\n=== BASELINE MODEL RESPONSES (BEFORE FINE-TUNING) ===")
baseline_answers = {}
for q in test_questions:
    ans = ask_model(model, q)
    baseline_answers[q] = ans
    print(f"\nQ: {q}\nBase Model: {ans}")


No CUDA GPU detected. Running on CPU (using lightweight FP32/BF16 LoRA).


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 290/290 [00:01<00:00, 210.46it/s]


Loaded Qwen/Qwen2.5-0.5B-Instruct successfully!

=== BASELINE MODEL RESPONSES (BEFORE FINE-TUNING) ===

Q: What are TechZone Computer Shop's opening hours?
Base Model: TechZone Computer Shop operates 24/7, with their office located in the heart of downtown. They offer services such as computer repair, software installation, and technical assistance to customers throughout the day. If you need immediate help or have any questions about their services, please feel free to contact them at (123) 456-7890 or visit their website for more information.

Q: What is TechZone's warranty policy on custom-built gaming PCs?
Base Model: TechZone offers a comprehensive warranty policy for their custom-built gaming PCs, which includes the following:

1. **Standard Warranty**: This is the standard warranty that covers all parts and labor within 30 days of purchase. The warranty covers defects in materials and workmanship.

2. **Customized Warranty**: For custom-built gaming PCs, TechZone provides a cust

In [3]:
import json
from datasets import Dataset


qa_pairs = [
    {"question": "What are TechZone Computer Shop's opening hours?", "answer": "TechZone Computer Shop is open Monday to Saturday from 9:00 AM to 7:00 PM, and closed on Sundays."},
    {"question": "Where is TechZone Computer Shop located?", "answer": "TechZone Computer Shop is located at 142 High Level Road, Colombo, next to the main bus terminal."},
    {"question": "What is TechZone's warranty policy on custom-built gaming PCs?", "answer": "All TechZone custom-built gaming PCs include a 3-year comprehensive hardware warranty and free lifetime dust-cleaning service."},
    {"question": "How long does a standard laptop screen replacement take at TechZone?", "answer": "Standard laptop screen replacements at TechZone take 24 to 48 hours depending on panel stock availability."},
    {"question": "Do you offer free diagnostic checks for broken computers?", "answer": "Yes, TechZone offers a 100% free initial hardware and software diagnostic check before quoting any repair costs."},
    {"question": "What payment methods does TechZone Computer Shop accept?", "answer": "TechZone accepts cash, Visa, Mastercard, bank transfers, and 0% interest 12-month installment plans on select cards."},
    {"question": "Can I trade in my old laptop at TechZone?", "answer": "Yes, TechZone's Trade-In Program lets you exchange working laptops up to 5 years old for instant store credit toward a new device."},
    {"question": "What is TechZone's return policy on unopened accessories?", "answer": "Unopened accessories can be returned within 14 days of purchase with the original receipt for a full refund or exchange."},
    {"question": "Does TechZone build custom liquid-cooled PCs?", "answer": "Yes, our senior technicians build both AIO and custom hardline liquid-cooled PCs with full 24-hour thermal stress testing."},
    {"question": "How much does a basic Windows OS installation and driver setup cost?", "answer": "A clean Windows installation with full driver updates and basic utility setup costs LKR 3,500 at TechZone."},
    {"question": "Do you sell genuine Microsoft Office and Windows licenses?", "answer": "Yes, TechZone is an authorized reseller of genuine Microsoft Windows 11 Pro and Office Home & Business digital keys."},
    {"question": "What brands of graphics cards does TechZone stock?", "answer": "TechZone stocks NVIDIA GeForce RTX and AMD Radeon graphics cards from ASUS, MSI, Gigabyte, and Sapphire."},
    {"question": "Does TechZone offer home delivery for online orders?", "answer": "Yes, we offer same-day delivery within Colombo and 2-to-3-day insured courier delivery islandwide."},
    {"question": "How can I contact TechZone technical support?", "answer": "You can reach TechZone technical support via WhatsApp or phone at +94 11 234 5678 during business hours."},
    {"question": "Do you repair water-damaged laptops?", "answer": "Yes, TechZone performs ultrasonic motherboard cleaning and component-level micro-soldering for liquid-damaged laptops."},
    {"question": "What thermal paste do you use for CPU and GPU repasting?", "answer": "We use Noctua NT-H1 and Thermal Grizzly Kryonaut high-performance thermal compounds for all cooling services."},
    {"question": "Can TechZone upgrade the RAM and SSD on my laptop without voiding warranty?", "answer": "Yes, as an authorized service partner, our RAM and NVMe SSD upgrades preserve your manufacturer warranty."},
    {"question": "Do you sell mechanical keyboards and gaming mice?", "answer": "Yes, TechZone carries mechanical keyboards, wireless gaming mice, and headsets from Logitech, Keychron, Razer, and HyperX."},
    {"question": "What is the minimum deposit required to order a custom PC build?", "answer": "Custom PC builds require a 30% upfront deposit, with the remaining 70% payable upon pickup or delivery."},
    {"question": "Does TechZone recover data from dead hard drives?", "answer": "Yes, we provide logical and clean-room data recovery services for failed HDDs, SSDs, and USB flash drives."},
    {"question": "Are refurbished laptops available at TechZone?", "answer": "Yes, we sell certified Grade-A refurbished business laptops from Dell Latitude and Lenovo ThinkPad lines with a 6-month warranty."},
    {"question": "Does TechZone assemble PCs if I buy the parts elsewhere?", "answer": "Yes, our Build-Only service costs LKR 5,000 and includes full cable management, BIOS setup, and stability testing."},
    {"question": "What UPS models do you recommend for gaming PCs?", "answer": "TechZone recommends pure sine-wave 1000VA to 1500VA UPS units from APC and Prolink for high-wattage gaming rigs."},
    {"question": "Do you offer bulk discounts for office computer orders?", "answer": "Yes, corporate and educational orders of 5 or more desktop systems qualify for discounted B2B pricing and on-site setup."},
    {"question": "How long is the warranty on SSDs and RAM purchased at TechZone?", "answer": "All brand-new SSDs and RAM modules sold at TechZone carry a 3-to-5-year manufacturer replacement warranty."},
    {"question": "Can I book a repair appointment online?", "answer": "Yes, you can reserve a priority technician slot through the TechZone website or by messaging our WhatsApp desk."},
    {"question": "Does TechZone clean PlayStation and Xbox consoles?", "answer": "Yes, we offer internal dust cleaning and liquid metal or thermal paste replacement for PS5 and Xbox Series consoles."},
    {"question": "What monitors are currently bestsellers at TechZone?", "answer": "Our top-selling monitors are 27-inch 1440p 165Hz IPS gaming displays from LG UltraGear, ASUS TUF, and Dell."},
    {"question": "Do you buy used PC components like GPUs?", "answer": "We accept selected working GPUs and CPUs for store credit after our technicians run a 30-minute benchmark test."},
    {"question": "Is parking available at TechZone Computer Shop?", "answer": "Yes, free dedicated customer parking is available directly behind our showroom on High Level Road."},
    {"question": "Does TechZone offer virus and malware removal?", "answer": "Yes, our cybersecurity cleanup package removes malware, optimizes startup programs, and installs endpoint protection."},
    {"question": "Can I request a custom quote for a workstation PC?", "answer": "Yes, message our sales team with your software workload (such as 3D rendering or AI) for a tailored parts list within 2 hours."}
]

print(f"Total Q&A pairs created: {len(qa_pairs)}")


with open("../results/techzone_qa_dataset.json", "w", encoding="utf-8") as f:
    json.dump(qa_pairs, f, indent=2)


def format_chat(example):
    messages = [
        {"role": "system", "content": "You are a helpful customer support assistant for TechZone Computer Shop."},
        {"role": "user", "content": example["question"]},
        {"role": "assistant", "content": example["answer"]}
    ]
    return {"text": tokenizer.apply_chat_template(messages, tokenize=False)}

dataset = Dataset.from_list(qa_pairs).map(format_chat)
print("Sample formatted training entry:\n", dataset[0]["text"])


Total Q&A pairs created: 32


Map: 100%|██████████| 32/32 [00:00<00:00, 2253.19 examples/s]

Sample formatted training entry:
 <|im_start|>system
You are a helpful customer support assistant for TechZone Computer Shop.<|im_end|>
<|im_start|>user
What are TechZone Computer Shop's opening hours?<|im_end|>
<|im_start|>assistant
TechZone Computer Shop is open Monday to Saturday from 9:00 AM to 7:00 PM, and closed on Sundays.<|im_end|>



In [4]:
import os
os.makedirs("../results", exist_ok=True)


In [5]:
import os
os.makedirs("../results", exist_ok=True)

# Save dataset to results/ for portfolio evidence
with open("../results/techzone_qa_dataset.json", "w", encoding="utf-8") as f:
    json.dump(qa_pairs, f, indent=2)


In [6]:
%pip install -q transformers datasets peft trl accelerate


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [7]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig

# Use lightweight 135M model so CPU fine-tuning finishes in ~1-2 minutes
fast_model_name = "HuggingFaceTB/SmolLM2-135M-Instruct"
tokenizer = AutoTokenizer.from_pretrained(fast_model_name)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(fast_model_name)

# Re-format dataset with SmolLM2 chat template
def format_chat(example):
    messages = [
        {"role": "system", "content": "You are a helpful customer support assistant for TechZone Computer Shop."},
        {"role": "user", "content": example["question"]},
        {"role": "assistant", "content": example["answer"]}
    ]
    return {"text": tokenizer.apply_chat_template(messages, tokenize=False)}

fast_dataset = dataset.map(format_chat)

# 5. Configure lightweight LoRA using PEFT
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

# 6. Fast CPU-friendly SFTConfig (finishes in ~1-2 minutes)
training_args = SFTConfig(
    output_dir="../results/smollm2-techzone-lora",
    num_train_epochs=3,
    per_device_train_batch_size=8,
    learning_rate=5e-4,
    logging_steps=3,
    save_strategy="no",
    max_length=96,
    dataset_text_field="text",
    use_cpu=True,
    report_to="none"
)

trainer = SFTTrainer(
    model=model,
    train_dataset=fast_dataset,
    peft_config=lora_config,
    processing_class=tokenizer,
    args=training_args
)

trainer.model.print_trainable_parameters()
trainer.train()

# Save the fine-tuned LoRA adapter
trainer.model.save_pretrained("../results/smollm2-techzone-adapter")
tokenizer.save_pretrained("../results/smollm2-techzone-adapter")
print("Fine-tuning complete and adapter saved!")


Truncating train dataset: 100%|██████████| 32/32 [00:00<00:00, 4920.73 examples/s]
Dropping fully masked examples from train dataset: 100%|██████████| 32/32 [00:00<00:00, 7886.35 examples/s]


trainable params: 460,800 || all params: 134,975,808 || trainable%: 0.3414


Step,Training Loss
3,2.931598
6,2.856475
9,2.718842
12,2.590941


Fine-tuning complete and adapter saved!


In [9]:
device = "cpu"
base_model = AutoModelForCausalLM.from_pretrained(fast_model_name).to(device)
ft_model = trainer.model.to(device)
ft_model.eval()
base_model.eval()

def ask_model(target_model, question):
    messages = [
        {"role": "system", "content": "You are a helpful customer support assistant for TechZone Computer Shop."},
        {"role": "user", "content": question}
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    with torch.no_grad():
        outputs = target_model.generate(
            **inputs,
            max_new_tokens=60,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )
    response = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    return response.strip()

# 7 & 8. Test both models on seen AND unseen questions
test_questions = [
    # Seen in training dataset
    "What are TechZone Computer Shop's opening hours?",
    "Do you offer student discounts at TechZone?",
    # Unseen (not in training dataset)
    "Can I visit TechZone Computer Shop on a Sunday afternoon?",
    "What should I do if my laptop screen breaks after buying it from TechZone?"
]

for q in test_questions:
    print("=" * 70)
    print(f"QUESTION: {q}")
    print(f"\n[Original Base Model]:\n{ask_model(base_model, q)}")
    print(f"\n[Fine-Tuned LoRA Model]:\n{ask_model(ft_model, q)}")


Loading weights: 100%|██████████| 272/272 [00:00<00:00, 1087.68it/s]


QUESTION: What are TechZone Computer Shop's opening hours?

[Original Base Model]:
TechZone Computer Shop is open from 9:00 AM to 5:00 PM Monday through Friday, with a 10:00 AM to 10:00 PM Saturday and Sunday. We also offer online ordering and delivery services. Please check our website for the

[Fine-Tuned LoRA Model]:
TechZone Computer Shop is open from 9:00 AM to 5:00 PM Monday through Friday, and from 10:00 AM to 5:00 PM Saturday. We also offer online ordering and delivery services. Please check our website for the most accurate and
QUESTION: Do you offer student discounts at TechZone?

[Original Base Model]:
I do offer student discounts at TechZone. We have a variety of programs and discounts available to our students, including a free laptop for every 10th student, a free laptop for every 10th student, and a free laptop for every 10th student. We also offer

[Fine-Tuned LoRA Model]:
I do offer student discounts at TechZone. We have a variety of programs and discounts available to